# Lab 1 · Part 2 — CSI Camera and Live Control

**Goal:** Learn  JetRacer camera


At the end of this exercise, you must be able to:

- Explain what a CSI camera does.
- Explain the difference between the original image and the wide image.
- Start, stop, and release the camera.
- Use the wide camera image and the saved car calibration together.

This exercise uses **`jetracer_calibration.json`** from Part 1. 

## 1. Prepare the JetRacer

1. Put the car on a stand. Keep the drive wheels above the table.
2. Check that the CSI ribbon cable is connected.
3. Close other notebooks or programs that use the camera.
4. Put `jetracer_calibration.json` in the same folder as this notebook.

**Safety:** Keep the car on the stand when you test the widgets. Keep the car's power switch within reach. A software STOP button is not an emergency stop.

## 2. What is a CSI camera?

The JetRacer uses an **IMX219 CSI camera**.

- **CSI** means *Camera Serial Interface*. It connects the camera sensor to the Jetson Nano.
- The camera has **one image sensor**. It is a monocular camera.
- The camera sends images to the Jetson. We will use these images for line detection and object detection.
- **NVIDIA Argus** manages the CSI camera. **GStreamer** transfers frames to Python.
- **JetCam** provides simple Python commands to read the frames.

Image path:

`IMX219 sensor → CSI connection → Argus / GStreamer → JetCam → Jupyter image widget`

### Key terms

| Term | Meaning |
|---|---|
| Frame | One image from the camera. |
| Resolution | Image width and height in pixels, such as `640 × 360`. |
| FPS | Frames per second. `30 FPS` means up to 30 frames each second. |
| Capture resolution | Image size requested from the camera pipeline. |
| Output resolution | Image size returned by JetCam after conversion or scaling. |
| Aspect ratio | Image shape. `1:1` is square. `16:9` is widescreen. |

## 3. Original camera view and wide camera view

NVIDIA's beginner notebook creates a camera with **224 × 224 output**:

```python
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224)
camera.running = True
```

This creates a **square image**. JetCam uses its default capture parameters when they are not specified.

For our JetRacer, we use this **wide view**:

```python
camera = CSICamera(
    width=640,
    height=360,
    capture_width=1280,
    capture_height=720,
    capture_fps=30
)
camera.running = True
```

| Setting | Original example | Wide view for this lab |
|---|---|---|
| Output image | `224 × 224` | `640 × 360` |
| Output shape | Square, `1:1` | Widescreen, `16:9` |
| Capture size | JetCam default | `1280 × 720` |
| Requested FPS | JetCam default | `30` |

**Important:** A wider *image shape* is not the same as a wider *lens field of view*. The camera lens is unchanged. The capture mode and scaling can change which parts of the scene are visible.

**From this point forward, use only the wide (`640 × 360`) configuration.** You do not need to open both camera modes at the same time.

## 4. Start, stop, and release the camera

JetCam can read a single image or capture images continuously.

| Command | Meaning |
|---|---|
| `camera.read()` | Read one frame directly when continuous capture is off and the camera is open. |
| `camera.running = True` | Start continuous capture in a background thread. |
| `camera.value` | Get the latest frame from continuous capture. |
| `camera.running = False` | Stop continuous capture. The camera connection remains open. |
| `camera.cap.release()` | Release the camera connection so another program can open it. |

**Do not use `camera.read()` while `camera.running` is `True`.** JetCam raises `RuntimeError: Cannot read directly while camera is running`. Use `camera.value` instead.

### Example: read one wide frame

```python
camera = CSICamera(
    width=640, height=360,
    capture_width=1280, capture_height=720, capture_fps=30
)
frame = camera.read()       # Read one frame.
camera.cap.release()        # Release the camera.
```

### Example: continuous wide camera

```python
camera = CSICamera(
    width=640, height=360,
    capture_width=1280, capture_height=720, capture_fps=30
)
camera.running = True      # Start continuous capture.
frame = camera.value       # Get the latest frame.
camera.running = False     # Stop continuous capture.
camera.cap.release()       # Release the camera.
```

These are **examples**. Do not run both at the same time. The **Start Camera** and **Stop Camera** buttons below will manage the camera connection for you. You can press Start again after Stop.

## 5. Load the saved car settings

The camera and the motor controller are separate devices. We will use the saved steering and throttle values from Part 1.

Run the cell below. It loads the JSON file and initializes the car with throttle set to **0**.

In [1]:
import json
from pathlib import Path

import cv2
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar

CONFIG_FILE = Path('jetracer_calibration.json')

if not CONFIG_FILE.exists():
    raise FileNotFoundError('Copy jetracer_calibration.json into this folder first.')

with CONFIG_FILE.open('r') as f:
    settings = json.load(f)

car = NvidiaRacecar()
car.throttle = 0.0
car.steering_gain = float(settings['steering_gain'])
car.steering_offset = float(settings['steering_offset'])
car.throttle_gain = float(settings['throttle_gain'])
car.steering = 0.0
car.steering_motor.throttle = car.steering_offset  # Apply saved center.

camera = None

print('Car ID:', settings['car_id'])
print('Steering gain:', car.steering_gain)
print('Steering offset:', car.steering_offset)
print('Throttle gain:', car.throttle_gain)
print('Throttle: 0.0 (STOP)')


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Car ID: jetracer_1
Steering gain: -0.9
Steering offset: 0.09
Throttle gain: 0.8
Throttle: 0.0 (STOP)


## 6. Build the car-control widgets

The dashboard will have:

- A **steering slider**. It uses the steering limits saved in JSON.
- A **throttle slider**. It uses the low test range from Part 1.
- An **Enable drive** checkbox. It is **off** when the notebook starts.
- A **STOP CAR** button.

You can steer while the car is stationary. The motor will receive a throttle command only when the camera is running **and** Enable drive is ticked.

Start with the car's wheels above the table.

In [2]:
steering_slider = widgets.FloatSlider(
    value=0.0,
    min=float(settings['steering_min']),
    max=float(settings['steering_max']),
    step=0.05,
    description='Steering',
    continuous_update=False,
    style={'description_width': 'initial'}
)

throttle_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=min(0.15, float(settings['normal'])),
    step=0.01,
    description='Throttle',
    continuous_update=False,
    style={'description_width': 'initial'}
)

enable_drive = widgets.Checkbox(value=False, description='Enable drive')
stop_car_button = widgets.Button(description='STOP CAR', button_style='danger')

def update_car(change=None):
    car.steering = steering_slider.value

    if enable_drive.value and camera is not None and camera.running:
        car.throttle = throttle_slider.value
    else:
        car.throttle = 0.0


def stop_car(button):
    enable_drive.value = False
    throttle_slider.value = 0.0
    car.throttle = 0.0

steering_slider.observe(update_car, names='value')
throttle_slider.observe(update_car, names='value')
enable_drive.observe(update_car, names='value')
stop_car_button.on_click(stop_car)


## 7. Build the camera controls

This dashboard uses **only the wide view**:

- Output: `640 × 360` pixels.
- Capture: `1280 × 720` pixels at `30 FPS`.
- **Start Camera:** Open the CSI camera and begin continuous capture.
- **Stop Camera:** Stop capture and release the camera. This button also stops and disarms the car.

The live frame is stored in `camera.value`. We convert it to JPEG for display in Jupyter. We display every third frame to reduce the load on the Jetson Nano.

In [3]:
start_camera_button = widgets.Button(
    description='Start Camera', button_style='success'
)
stop_camera_button = widgets.Button(
    description='Stop Camera', button_style='warning', disabled=True
)

camera_status = widgets.HTML(value='<b>Camera:</b> Stopped')
live_image = widgets.Image(format='jpeg', width=640, height=360)
frame_count = 0


def show_frame(change):
    global frame_count
    frame_count += 1

    if frame_count % 3 != 0:
        return

    ok, image = cv2.imencode('.jpg', change['new'])
    if ok:
        live_image.value = image.tobytes()


def start_camera(button):
    global camera, frame_count
    if camera is not None:
        return

    camera_status.value = '<b>Camera:</b> Opening...'
    start_camera_button.disabled = True
    frame_count = 0

    try:
        camera = CSICamera(
            width=640,
            height=360,
            capture_width=1280,
            capture_height=720,
            capture_fps=30
        )
        camera.observe(show_frame, names='value')
        camera.running = True

    except Exception as error:
        if camera is not None:
            if camera.running:
                camera.running = False
            camera.cap.release()
        camera = None
        start_camera_button.disabled = False
        camera_status.value = '<b>Camera:</b> Error: ' + str(error)
        return

    stop_camera_button.disabled = False
    camera_status.value = '<b>Camera:</b> Running (wide 640 x 360)'


def stop_camera(button):
    global camera
    stop_car(None)  # Stop and disarm the drive motor.

    if camera is not None:
        camera.running = False
        camera.unobserve(show_frame, names='value')
        camera.cap.release()
        camera = None

    live_image.value = b''
    start_camera_button.disabled = False
    stop_camera_button.disabled = True
    camera_status.value = '<b>Camera:</b> Stopped and released'

start_camera_button.on_click(start_camera)
stop_camera_button.on_click(stop_camera)

## 8. Run the camera and car dashboard

Run the next cell to display the dashboard.


**Exercise — Control the car**

1. Keep the drive wheels above the table.
2. Move the **Steering** slider. Check the wheel direction.
3. Return Steering to **0.00**.
4. Set Throttle to **0.11** or **0.12**. These are the low-speed values from Part 1.
5. Tick **Enable drive** only when the instructor allows the test.
6. Press **STOP CAR** after the test.
7. Press **Stop Camera** to release the CSI camera.

*Do not drive the car on the floor while looking only at the live stream. A preview can freeze or be delayed.*

In [4]:
camera_panel = widgets.VBox([
    widgets.HTML('<h3>Wide CSI Camera (640 x 360)</h3>'),
    widgets.HBox([start_camera_button, stop_camera_button]),
    camera_status,
    live_image
])

car_panel = widgets.VBox([
    widgets.HTML('<h3>Car Control</h3>'),
    steering_slider,
    throttle_slider,
    enable_drive,
    stop_car_button
])

display(widgets.VBox([camera_panel, car_panel]))

## 9. Inspect one frame

When the camera runs, `camera.value` stores the latest image. A color image has three values in its `shape`:

`(height, width, color channels)`

Run the next cell while the wide camera is running. The expected shape is **`(360, 640, 3)`**.

In [5]:
if camera is None or not camera.running:
    print('Start the camera first.')
else:
    frame = camera.value
    print('Frame shape:', frame.shape)
    print('Camera running:', camera.running)


Start the camera first.


## 10. Troubleshooting: when the camera does not start

The CSI camera is managed by a system service called **`nvargus-daemon`**. This service is separate from your notebook.

### First, check the notebook

1. Press **Stop Camera** if the button is active.
2. Close any other notebook, OpenCV script, or GStreamer test that uses the camera.
3. Shut down the old Jupyter kernel if it still holds the camera.
4. Start the camera again.

### If the camera still fails

Open a **Jetson Nano terminal**, not a Python cell. Run:

```bash
sudo systemctl status nvargus-daemon
sudo systemctl restart nvargus-daemon
sudo systemctl status nvargus-daemon
```

- `status` shows whether the service is active.
- `restart` stops and starts the Argus service. It **does not reboot the Jetson**.
- Restart the service **only after** all camera applications have stopped.
- Do not run `systemctl` commands in a Python code cell.

If you still cannot open the camera, check the device and recent logs:

```bash
ls /dev/video*
sudo journalctl -u nvargus-daemon -n 30 --no-pager
```

If `ls /dev/video*` shows no device, check the camera cable and hardware connection with the car powered off. Ask the instructor before reconnecting a ribbon cable.

**Common mistake:** Calling `camera.read()` while `camera.running` is `True` raises `RuntimeError: Cannot read directly while camera is running`. Use `camera.value` instead.

## 11. Finish the exercise

1. Press **STOP CAR**.
2. Press **Stop Camera**. This sets `camera.running` to `False` and releases the connection.
3. Confirm that the camera status says **Stopped and released**.
4. Shut down the notebook kernel before another notebook opens the camera.
5. Switch the car off when the instructor tells you to do so.

**Check your understanding:**

- What is the output resolution of the original NVIDIA camera example?
- What capture resolution and output resolution do we use for the wide view?
- What is the difference between `camera.running = False` and `camera.cap.release()`?
- Why does `camera.value` work during continuous capture, but `camera.read()` does not?
- Which camera view will you use in the next lab?

**Result:** You can operate the CSI camera and control the JetRacer using the saved calibration values. The next exercise will use **wide output (`640 × 360`)** for image processing.

In [ ]:
# Run this cell before you leave the notebook.
stop_camera(None)
car.steering = 0.0
print('Car stopped. Camera released. Exercise complete.')


In [6]:
!echo $USER | sudo -S systemctl restart nvargus-daemon 

[sudo] password for jetson: 